# Generate clinical note embeddings (GPU, GCP)

Self-contained driver for `generate_clinical_embeddings.py` on a GCP GPU VM/notebook.
It assumes the repository is cloned at `/home/patrickconnor/clinical_text_embedding_project`
and `batched_tokens` has already been copied to `/home/patrickconnor/data/batched_tokens`.
The notebook does not move data to or from the cluster. The underlying script only depends on
torch/numpy/transformers/flash-attn/tqdm/zstandard.

Steps: install dependencies, verify a GPU is visible, configure the local paths, and run the
embedding script. Results are written to `/home/patrickconnor/data/embeddings`.

In [ ]:
%pip install -q torch numpy transformers tqdm zstandard ninja packaging
%pip install -q flash-attn --no-build-isolation

## Verify GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No CUDA device visible - check the GCP VM/notebook GPU runtime."
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configure paths

The default layout is shown below. Environment variables remain explicit so the paths passed to
the embedding subprocess are unambiguous.

In [ ]:
import os
from pathlib import Path

HOME_ROOT = Path("/home/patrickconnor")
REPO_ROOT = HOME_ROOT / "clinical_text_embedding_project"
DATA_ROOT = HOME_ROOT / "data"
BATCHED_TOKENS_PATH = DATA_ROOT / "batched_tokens"

os.environ["CLINICAL_EMBED_DATA_ROOT"] = str(DATA_ROOT)
os.environ["BATCHED_TOKENS_PATH"] = str(BATCHED_TOKENS_PATH)
os.environ["TOKEN_PATH"] = str(BATCHED_TOKENS_PATH / "tokens")
os.environ["EMBED_PATH"] = str(DATA_ROOT / "embeddings")
os.environ["EMBED_MAX_BATCH_SIZE"] = "128"
os.environ["EMBED_MAX_ATTENTION_ELEMENTS"] = str(8192**2)
os.environ["EMBED_COLLATE_WORKERS"] = "2"
os.environ["EMBED_COLLATE_PREFETCH_FACTOR"] = "2"
os.environ["EMBED_ZSTD_LEVEL"] = "3"
os.environ["EMBED_COMPILE"] = "1"
os.environ["EMBED_COMPILE_MODE"] = "default"

Path(os.environ["EMBED_PATH"]).mkdir(parents=True, exist_ok=True)

if not Path(os.environ["TOKEN_PATH"]).is_dir():
    raise FileNotFoundError(f"Token directory not found: {os.environ['TOKEN_PATH']}")

print(f"REPO_ROOT:  {REPO_ROOT}")
print(f"TOKEN_PATH: {os.environ['TOKEN_PATH']}")
print(f"EMBED_PATH: {os.environ['EMBED_PATH']}")

## Run embedding generation

In [ ]:
import sys

SCRIPT_PATH = REPO_ROOT / "pipelines" / "preprocessing" / "generate_clinical_embeddings.py"

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"generate_clinical_embeddings.py not found at {SCRIPT_PATH}. "
        f"Confirm the repository is cloned at {REPO_ROOT}."
    )

subprocess.run(
    [
        sys.executable, str(SCRIPT_PATH),
        "--smoke-test-notes", "8",
        "--run-after-smoke-test",
    ],
    check=True,
)
print(f"Done. Embeddings written to {os.environ['EMBED_PATH']}")